In [0]:
from pyspark.sql import functions as F

# ================== Step 1: Config ==================
source_path = "s3://ujjivanpoc/source/bank_transactions/"
target_path = "s3://ujjivanpoc/Bronze/bank_transaction_fraud_detection"
target_table = "ujjivan_2.bronze.bank_transaction_fraud_detection"
checkpoint_path = "s3://ujjivanpoc/Bronze/_checkpoints/bank_transaction_fraud_detection"
schema_location = "s3://ujjivanpoc/Bronze/_schemas/bank_transaction_fraud_detection"

# IMPORTANT: Keep this False for normal/incremental runs.
# Autoloader uses the checkpoint + schema location to remember which files
# it has already ingested. Wiping them (True) forces it to treat every file
# in source_path as brand new again, causing full re-ingestion/duplicates.
# Only flip this to True intentionally — e.g. one-off POC reset, or a
# deliberate full reload after a major schema change — then set it back
# to False immediately after that single run.
RESET_CHECKPOINT = False

# ================== Step 2: Create the empty table ONLY if it doesn't already exist ==================
if not spark.catalog.tableExists(target_table):
    schema_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
    )
    (
        schema_df.write
        .format("delta")
        .mode("overwrite")
        .option("path", target_path)
        .saveAsTable(target_table)
    )
    print(f"✅ Created table {target_table} at {target_path}")
else:
    print(f"ℹ️ Table {target_table} already exists — proceeding with incremental load")

# ================== Step 3: Optionally reset Autoloader state ==================
# Only runs if RESET_CHECKPOINT is explicitly set to True above.
if RESET_CHECKPOINT:
    dbutils.fs.rm(checkpoint_path, recurse=True)
    dbutils.fs.rm(schema_location, recurse=True)
    print("Cleared checkpoint and schema location — forcing full re-discovery of source files")
else:
    print("Checkpoint/schema preserved — this run will only pick up new/unseen files")

# ================== Step 4: Row count BEFORE load ==================
count_before = spark.table(target_table).count()
print(f"📊 Row count BEFORE load: {count_before}")

# ================== Step 5: Incremental load via Autoloader ==================
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # catches malformed/unmatched columns
    .load(source_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

query = (
    df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

# ================== Step 6: Block until this batch actually finishes ==================
query.awaitTermination()

# ================== Step 7: Row count AFTER load ==================
count_after = spark.table(target_table).count()
rows_inserted = count_after - count_before

print(f"📊 Row count AFTER load: {count_after}")
print(f"📈 Rows inserted this run: {rows_inserted}")

# ================== Step 8: Micro-batch level detail ==================
progress_list = query.recentProgress
if not progress_list:
    print("⚠️ No progress recorded — query may not have processed any batches (no new files found).")
else:
    print(f"\n🔍 Processed {len(progress_list)} micro-batch(es):")
    total_input_rows = 0
    for i, p in enumerate(progress_list, start=1):
        num_input_rows = p.get("numInputRows", 0)
        total_input_rows += num_input_rows
        source = p.get("sources", [{}])[0]
        print(
            f"  Batch {i}: "
            f"inputRows={num_input_rows}, "
            f"batchDuration={p.get('batchDuration')}ms, "
            f"latestOffset={source.get('latestOffset')}, "
            f"description={source.get('description', 'N/A')[:60]}"
        )
    print(f"\n📥 Total input rows read from source across all batches: {total_input_rows}")

# ================== Step 9: Check for rescued/malformed rows ==================
if "_rescued_data" in spark.table(target_table).columns:
    rescued_count = spark.table(target_table).filter(F.col("_rescued_data").isNotNull()).count()
    if rescued_count > 0:
        print(f"⚠️ {rescued_count} row(s) had rescued/malformed data — check the '_rescued_data' column")

print(f"\n✅ Incremental load complete into {target_table} — {rows_inserted} new row(s) inserted")